### 项目结构

```
mcp-homework/
├── Dockerfile
├── docker-compose.yml
├── server.py
└── requirements.txt
```

---

## 第一步：创建项目文件

**requirements.txt：**
```
mcp
```

**server.py：**
```python
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent
import asyncio
import json

app = Server("my-mcp-server")

@app.list_tools()
async def list_tools():
    return [
        Tool(
            name="say_hello",
            description="向指定的人打招呼",
            inputSchema={
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "人的名字"
                    }
                },
                "required": ["name"]
            }
        ),
        Tool(
            name="calculate",
            description="执行简单的数学计算",
            inputSchema={
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，比如 1+2*3"
                    }
                },
                "required": ["expression"]
            }
        )
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "say_hello":
        person = arguments.get("name", "世界")
        return [TextContent(type="text", text=f"你好，{person}！我是运行在 Docker 里的 MCP 服务！")]

    elif name == "calculate":
        expr = arguments.get("expression", "")
        try:
            result = eval(expr)
            return [TextContent(type="text", text=f"{expr} = {result}")]
        except Exception as e:
            return [TextContent(type="text", text=f"计算错误：{str(e)}")]

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
```

**Dockerfile：**
```dockerfile
FROM docker.m.daocloud.io/library/python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

COPY server.py .

CMD ["python", "server.py"]
```

**docker-compose.yml：**
```yaml
services:
  mcp-server:
    build: .
    container_name: my_mcp_service
    stdin_open: true
    tty: true
    restart: unless-stopped
```

---

## 第二步：构建并启动

```powershell
cd mcp-homework
docker compose up -d --build
```

---

## 第三步：Cherry Studio 配置 MCP

打开 Cherry Studio → 设置 → MCP 服务器 → 添加：

| 配置项 | 填写值 |
|--------|--------|
| 名称 | my-mcp-server |
| 类型 | stdio |
| 命令 | `docker` |
| 参数 | `exec -i my_mcp_service python server.py` |

---

## 第四步：验证

在 Cherry Studio 对话中，AI 应该能调用 `say_hello` 和 `calculate` 两个工具。

---

先把这几个文件创建好，然后把构建结果发给我，我们一步步完成！